# Fink LSST — SALT2 Fit with Free Redshift on TNS SNIa

- **author** : Sylvie Dagoret-Campagne
- **affiliation** : IJCLab/IN2P3/CNRS — Université Paris-Saclay
- **creation date** : 2026-05-10
- **last update**: 2026-05-12 : implement error calculation on distance modulus
- **based on** : `07_TNS_SN/03_fink_tns_sn_fitSNIa_salt2_lightcurves.ipynb` (z fixed to TNS)
- **strategy from** : `02_sncosmo/02b_elasticc2_fitsalt2andredshift_lightcurves.ipynb` (ELAsTiCC2 free-z)

## Purpose

This notebook fits Fink/LSST TNS SNIa light curves with SALT2 treating the
**redshift as a free parameter** (photometric redshift from the light-curve shape).

The TNS spectroscopic redshift `f:xm_tns_redshift` is used **only for validation**
at the end — it is never fed to the fitter.

Free parameters: $\{z,\; t_0,\; x_0,\; x_1,\; c\}$

## Two-pass redshift strategy

Fitting z from broadband photometry alone is degenerate (z–c degeneracy).  
We use the same two-pass strategy as `02b_elasticc2_fitsalt2andredshift`:

1. **Coarse grid scan** — scan `N_GRID_Z` redshift values in `[Z_BOUND_LO, Z_BOUND_HI]`,
   fix `z`, fit `(t0, x0, x1, c)`. Record χ².
2. **Refined free-z fit** — initialise all 5 parameters at the grid minimum;
   free `z` within a window of ±`Z_REFINE_HALF` around the best grid point.
   The covariance matrix yields `σ_z`.

## Final diagnostics

- **Hubble diagram** μ(z_fit) and μ(z_TNS) vs ΛCDM.
- **Photometric z accuracy**: `z_fit ± σ_z` vs `z_TNS` (the spectroscopic reference),
  residuals Δz and Δz/(1+z_TNS).

### Column conventions
- `r:<col>` — LSST diaSource/diaObject field (prefix = table name, NOT r-band)
- `f:<col>` — Fink-computed field
- Flux: `r:psfFlux` in **nJy**, AB ZP = 31.4

### References
- sncosmo: https://sncosmo.readthedocs.io
- Guy et al. 2007 (SALT2); Betoule et al. 2014 (Tripp formula)
- Notebook `03_fink_tns_sn_fitSNIa_salt2_lightcurves.ipynb` — z-fixed baseline
- Notebook `02b_elasticc2_fitsalt2andredshift_lightcurves.ipynb` — ELAsTiCC2 free-z


## 0 — Imports

In [ ]:
import io
import math
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import astropy.table
import matplotlib
import matplotlib.pyplot as plt
import requests
import sncosmo
from scipy.integrate import quad

plt.rcParams.update({"figure.dpi": 120, "font.size": 10})
print("Imports OK")

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → %matplotlib inline")

## 1 — Configuration

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
# Data produced by notebook 01_fink_tns_sn_lightcurves.ipynb
NB01_DATA_DIR = Path("data_NB07_01_TNS_SN")
CATALOG_FILE = NB01_DATA_DIR / "catalog_fink_in_tns_lsst.parquet"

# Output directories for this notebook
NB_TAG = "NB07_04_FINK_TNS_SN_FIT_SALT2_FREEZ"
DATA_DIR = Path(f"data_{NB_TAG}")
FIGS_DIR = Path(f"figs_{NB_TAG}")
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGS_DIR.mkdir(parents=True, exist_ok=True)

# ── Fink API ──────────────────────────────────────────────────────────────────
FINK_API = "https://api.lsst.fink-portal.org/api/v1"
API_DELAY = 0.4

# ── Photometric system ────────────────────────────────────────────────────────
RUBIN_ZP = 31.4
ZPSYS = "ab"

# ── SALT2 model ───────────────────────────────────────────────────────────────
SALT2_SOURCE = "salt2-extended"
BAND_PREFIX = "lsst"
BAND_ORDER = ["u", "g", "r", "i", "z", "y"]

# ── Object selection ──────────────────────────────────────────────────────────
# TNS spec-z is used for VALIDATION only — not passed to the fitter
Z_MIN_SEL = 0.01  # selection cut on TNS z (to have a validation reference)
Z_MAX_SEL = 1.5
MIN_DATAPOINTS = 5  # minimum SNR-passing data points required
SNR_MIN = 3.0

# SNIa type strings in TNS (case-insensitive partial match)
SNIa_TYPES = ["SN Ia", "SN Ia-pec", "SN Ia-91T", "SN Ia-91bg", "SN I", "SN Ic-BL"]
SN_TYPES = ["SN Ia", "SN I", "SN Ib", "SN Ic", "SN II", "SN IIn", "SN Ic-BL", "SLSN-I"]
SN_MARKERS = ["o", "+", "x", "1", "2", "3", "4", "X"]
SN_MARKERS2 = ["o", "s", "D", "^", "v", "<", ">", "X"]

# Set to None to attempt fit on ALL objects with a redshift (including unclassified)
RESTRICT_TO_SNIa = False  # set False to attempt fit on all types

# ── Two-pass redshift fit ─────────────────────────────────────────────────────
# Agnostic survey-level bounds — TNS z NOT used to define these
Z_BOUND_LO = 0.02  # SALT2-extended + LSST u-band practical limit
Z_BOUND_HI = 1.50
N_GRID_Z = 100  # coarse grid points
Z_REFINE_HALF = 0.15  # half-width of the refinement window around grid best-z

# ── Tripp formula nuisance parameters ────────────────────────────────────────
ALPHA = 0.14
BETA = 3.14
M_B = -19.3

# ── ΛCDM cosmology ───────────────────────────────────────────────────────────
H0 = 70.0
OmegaM = 0.30
OmegaL = 0.70

# ── Plot colours per band ─────────────────────────────────────────────────────
BAND_COLORS = {
    "u": "#3498DB",
    "g": "#27AE60",
    "r": "#E74C3C",
    "i": "#F1C40F",
    "z": "#8B4513",
    "y": "#95A5A6",
}
NCOLS_LC = 3

print(f"SALT2 source  : {SALT2_SOURCE}")
print(f"ZP={RUBIN_ZP}  zpsys={ZPSYS}")
print(f"Redshift fit  : z ∈ [{Z_BOUND_LO}, {Z_BOUND_HI}]  (NO TNS z used in fit)")
print(f"Grid scan     : {N_GRID_Z} pts  |  refine ±{Z_REFINE_HALF}")
print(f"Tripp α={ALPHA}  β={BETA}  M_B={M_B}")
print(f"ΛCDM H0={H0}  Ωm={OmegaM}  ΩΛ={OmegaL}")

## 2 — Load catalog and select objects with TNS spec-z

The TNS spectroscopic redshift is used **only** to select the validation sample
and for the final z_fit vs z_true diagnostic — it is **never** passed to the fitter.

In [ ]:
if not CATALOG_FILE.exists():
    raise FileNotFoundError(
        f"Catalog not found: {CATALOG_FILE}\nPlease run notebook 01_fink_tns_sn_lightcurves.ipynb first."
    )

catalog = pd.read_parquet(CATALOG_FILE)
print(f"Catalog loaded: {len(catalog)} objects")

tns_z_col = "f:xm_tns_redshift"
tns_type_col = "f:xm_tns_type"
tns_name_col = "f:xm_tns_fullname"
obj_id_col = "r:diaObjectId"

has_z = catalog[tns_z_col].notna() & (catalog[tns_z_col] > 0)
in_range = (catalog[tns_z_col] >= Z_MIN_SEL) & (catalog[tns_z_col] <= Z_MAX_SEL)
sel = has_z & in_range

if RESTRICT_TO_SNIa:
    is_snia = (
        catalog[tns_type_col].fillna("").apply(lambda t: any(s.lower() in t.lower() for s in SNIa_TYPES))
    )
    sel = sel & is_snia

snia_catalog = catalog[sel].copy().reset_index(drop=True)
print(f"Objects with valid TNS z in [{Z_MIN_SEL}, {Z_MAX_SEL}]: {has_z.sum()}")
print(f"After {'SNIa type filter' if RESTRICT_TO_SNIa else 'no type filter'}: {len(snia_catalog)}")

# Show the validation sample
snia_catalog[[obj_id_col, tns_name_col, tns_type_col, tns_z_col]].head(25)

## 3 — Load or download light curves

Triple fallback cache: NB04 local → NB01 `light_curves/` subdir → Fink API (`diaObjectId`).

In [ ]:
LC_COLUMNS = ",".join(
    [
        "r:diaObjectId",
        "r:diaSourceId",
        "r:midpointMjdTai",
        "r:band",
        "r:psfFlux",
        "r:psfFluxErr",
        "r:snr",
        "r:reliability",
    ]
)


def fetch_lc(obj_id: int, cache_dir: Path, delay: float = API_DELAY) -> pd.DataFrame:
    """Load (from cache) or download the Fink LSST alert history for one diaObjectId.

    Cache priority
    --------------
    1. NB04 own cache: cache_dir / lc_{obj_id}.parquet
    2. NB01 cache:     NB01_DATA_DIR / light_curves / lc_{obj_id}.parquet
    3. Fink API:       /api/v1/sources?diaObjectId=...
    """
    cache_path = cache_dir / f"lc_{obj_id}.parquet"
    if cache_path.exists():
        return pd.read_parquet(cache_path)

    nb01_cache = NB01_DATA_DIR / "light_curves" / f"lc_{obj_id}.parquet"
    if nb01_cache.exists():
        df = pd.read_parquet(nb01_cache)
        df.to_parquet(cache_path, index=False)
        return df

    # API download — parameter is diaObjectId (NOT objectId)
    params = {"diaObjectId": obj_id, "columns": LC_COLUMNS, "output-format": "json"}
    try:
        resp = requests.get(f"{FINK_API}/sources", params=params, timeout=60)
        resp.raise_for_status()
        if not resp.text.strip():
            print(f"  [warn] {obj_id}: empty response")
            return pd.DataFrame()
        df = pd.read_json(io.BytesIO(resp.content))
        if "r:midpointMjdTai" in df.columns:
            df = df.sort_values("r:midpointMjdTai").reset_index(drop=True)
        df.to_parquet(cache_path, index=False)
        time.sleep(delay)
        return df
    except Exception as exc:
        print(f"  [error] {obj_id}: {exc}")
        return pd.DataFrame()


print(f"fetch_lc ready  →  cache: {DATA_DIR}/")

In [ ]:
lc_dict = {}

for _, row in snia_catalog.iterrows():
    oid = int(row[obj_id_col])
    name = row.get(tns_name_col, str(oid))
    df = fetch_lc(oid, DATA_DIR)
    lc_dict[oid] = df
    status = f"{len(df)} pts" if not df.empty else "EMPTY"
    print(f"  {name:22s}  diaObjectId={oid}  → {status}")

n_ok = sum(1 for df in lc_dict.values() if not df.empty)
print(f"\n{n_ok}/{len(lc_dict)} light curves loaded.")

## 4 — Helper: Fink DataFrame → sncosmo Table

psfFlux [nJy], ZP = 31.4, zpsys = 'ab'.  
Band prefix: `lsst` + single letter (lsstu, lsstg, …).

In [ ]:
def make_sncosmo_table_fink(
    lc_df: pd.DataFrame,
    snr_min: float = SNR_MIN,
    zp: float = RUBIN_ZP,
    zpsys: str = ZPSYS,
    band_prefix: str = BAND_PREFIX,
) -> astropy.table.Table:
    """Convert a Fink LSST alert DataFrame to an astropy.Table for sncosmo.

    Applies quality cuts: fluxerr > 0 and snr >= snr_min.
    Band names: 'r:band' single letter → 'lsst' + letter.
    """
    df = lc_df.rename(
        columns={
            "r:midpointMjdTai": "time",
            "r:band": "band_raw",
            "r:psfFlux": "flux",
            "r:psfFluxErr": "fluxerr",
            "r:snr": "snr",
        }
    ).copy()
    df = df[df["fluxerr"] > 0]
    if "snr" in df.columns:
        df = df[df["snr"] >= snr_min]
    df["band"] = band_prefix + df["band_raw"].str.strip().str.lower()
    return astropy.table.Table(
        {
            "time": df["time"].values.astype(float),
            "band": df["band"].values,
            "flux": df["flux"].values.astype(float),
            "fluxerr": df["fluxerr"].values.astype(float),
            "zp": np.full(len(df), zp, dtype=float),
            "zpsys": np.full(len(df), zpsys),
        }
    )


print("make_sncosmo_table_fink ready.")

## 5 — Two-pass SALT2 fitter with free redshift

**Pass 1 — coarse grid scan**  
Scan `N_GRID_Z` points in `[Z_BOUND_LO, Z_BOUND_HI]`. At each point fix `z`,
run a 4-parameter fit `(t0, x0, x1, c)`, record χ².

**Pass 2 — refined free-z fit**  
Initialise at `z_grid_best` and free all 5 parameters
`(z, t0, x0, x1, c)` within `[z_grid_best − Z_REFINE_HALF, z_grid_best + Z_REFINE_HALF]`.  
The covariance matrix gives `σ_z`.

In [ ]:
def fit_salt2_free_z_fink(lc_df: pd.DataFrame) -> dict:
    """Fit one Fink LSST SNIa light curve with SALT2, redshift free.

    TNS spectroscopic redshift is NOT used here.
    Two-pass strategy: coarse z-grid scan → refined 5-parameter free fit.

    Returns
    -------
    dict with keys:
        success, z, z_err, t0, x0, x1, c,
        chi2, ndof, chi2_red,
        mB, mu_tripp,
        z_grid, chi2_grid, z_grid_best,
        fitted_model, table, message
    """
    # ── Build sncosmo observation table ───────────────────────────────────────
    try:
        obs = make_sncosmo_table_fink(lc_df)
    except Exception as exc:
        return {"success": False, "message": f"Table build failed: {exc}"}

    if len(obs) < MIN_DATAPOINTS:
        return {
            "success": False,
            "message": f"Only {len(obs)} pts after quality cuts (need {MIN_DATAPOINTS})",
        }

    t0_guess = float(obs["time"][np.argmax(obs["flux"])])

    # ── Pass 1: coarse z grid scan ────────────────────────────────────────────
    z_grid = np.linspace(Z_BOUND_LO, Z_BOUND_HI, N_GRID_Z)
    chi2_grid = np.full(N_GRID_Z, np.inf)
    bounds_4p = {
        "t0": (t0_guess - 30.0, t0_guess + 30.0),
        "x0": (1e-10, 1e2),
        "x1": (-5.0, 5.0),
        "c": (-0.5, 0.5),
    }

    for k, z_try in enumerate(z_grid):
        m = sncosmo.Model(source=SALT2_SOURCE)
        m.set(z=z_try, t0=t0_guess, x0=1e-3, x1=0.0, c=0.0)
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                res_k, _ = sncosmo.fit_lc(
                    obs, m, vparam_names=["t0", "x0", "x1", "c"], bounds=bounds_4p, minsnr=0.0, warn=False
                )
            chi2_grid[k] = res_k.chisq
        except Exception:
            pass

    best_k = int(np.argmin(chi2_grid))
    z_grid_best = float(z_grid[best_k])

    # ── Pass 2: refined free-z fit ────────────────────────────────────────────
    z_ref_lo = max(Z_BOUND_LO, z_grid_best - Z_REFINE_HALF)
    z_ref_hi = min(Z_BOUND_HI, z_grid_best + Z_REFINE_HALF)

    model_final = sncosmo.Model(source=SALT2_SOURCE)
    model_final.set(z=z_grid_best, t0=t0_guess, x0=1e-3, x1=0.0, c=0.0)

    bounds_5p = {
        "z": (z_ref_lo, z_ref_hi),
        "t0": (t0_guess - 30.0, t0_guess + 30.0),
        "x0": (1e-10, 1e2),
        "x1": (-5.0, 5.0),
        "c": (-0.5, 0.5),
    }

    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            result, fitted_model = sncosmo.fit_lc(
                obs,
                model_final,
                vparam_names=["z", "t0", "x0", "x1", "c"],
                bounds=bounds_5p,
                minsnr=0.0,
                warn=False,
            )
    except Exception as exc:
        return {
            "success": False,
            "message": f"Pass-2 failed: {exc}",
            "z_grid": z_grid,
            "chi2_grid": chi2_grid,
            "z_grid_best": z_grid_best,
            "table": obs,
        }

    chi2 = float(result.chisq)
    ndof = int(result.ndof)
    chi2_red = chi2 / max(ndof, 1)

    # ── Covariance → σ_z ─────────────────────────────────────────────────────
    vpnames = ["z", "t0", "x0", "x1", "c"]
    vcov = result.covariance
    param_errors = {}
    if vcov is not None:
        diag = np.diag(vcov)
        for i, p in enumerate(vpnames):
            param_errors[p] = float(np.sqrt(max(diag[i], 0.0)))

    # ── Tripp distance modulus ────────────────────────────────────────────────
    x0_fit = float(fitted_model["x0"])
    x1_fit = float(fitted_model["x1"])
    c_fit = float(fitted_model["c"])
    SALT2_ZP_INT = 10.635
    mB = -2.5 * np.log10(max(x0_fit, 1e-30)) + SALT2_ZP_INT
    mu = mB - M_B + ALPHA * x1_fit - BETA * c_fit

    return {
        "success": True,
        "result": result,
        "fitted_model": fitted_model,
        "table": obs,
        "z": float(fitted_model["z"]),
        "z_err": param_errors.get("z", float("nan")),
        "t0": float(fitted_model["t0"]),
        "x0": x0_fit,
        "x1": x1_fit,
        "c": c_fit,
        "chi2": chi2,
        "ndof": ndof,
        "chi2_red": chi2_red,
        "vcov": vcov,
        "param_errors": param_errors,
        "z_grid": z_grid,
        "chi2_grid": chi2_grid,
        "z_grid_best": z_grid_best,
        "mB": mB,
        "mu_tripp": mu,
        "message": result.message,
    }


print("Two-pass SALT2 free-z fitter ready.")

In [ ]:
def computemustaterrors(
    param_names, params, covariance, mb_const=10.635, M=-19.3, alpha=0.14, beta=3.1, cosmo=None
):
    """Propagate SALT2 fit covariance into an uncertainty on the Tripp distance modulus.

    Tripp formula
    -------------
        mu  =  m_B*  -  M  +  alpha*x1  -  beta*c
        m_B*  =  -2.5 * log10(x0)  +  mb_const

    Partial derivatives of mu (gradient vector g):
        d(mu)/d(x0) = -2.5 / (x0 * ln10)
        d(mu)/d(x1) =  alpha
        d(mu)/d(c)  = -beta
        d(mu)/d(z)  = (5/ln10)*(1/d_L)*dd_L/dz   [only when z is a free parameter]

    The full error propagation is evaluated as:
        sigma_mu = sqrt( g^T . C_sub . g )
    where C_sub is the covariance sub-matrix restricted to the parameters
    that enter mu (x0, x1, c and optionally z).  t0 is excluded.

    Parameters
    ----------
    param_names : list[str]
        Names of *all* fitted parameters, in the same order as `params` and
        the rows/columns of `covariance`.
        z fixed : ['t0', 'x0', 'x1', 'c']
        z free  : ['z', 't0', 'x0', 'x1', 'c']
    params      : array-like  – fitted values, same order as param_names.
    covariance  : 2-D array   – full covariance matrix, same order.
    mb_const    : SALT2 zero-point offset (default 10.635 for salt2-extended).
    M           : absolute B-band magnitude of a standard SNIa.
    alpha       : Tripp stretch coefficient.
    beta        : Tripp colour coefficient.
    cosmo       : astropy cosmology instance; used only when z is free.
                  Defaults to FlatLambdaCDM(H0=70, Om0=0.3).

    Returns
    -------
    dict with keys:
        mu          – Tripp distance modulus
        sigma_mu    – 1-sigma uncertainty on mu
        gradient    – {param_name: d(mu)/d(param)} for mu-entering params
        sigmas      – {param_name: 1-sigma marginal error} for all params
        covariances – {'pi:pj': cov(pi,pj)} for mu-entering pairs
    """
    params = np.asarray(params, dtype=float)
    cov = np.asarray(covariance, dtype=float)
    names = list(param_names)

    # Build a name→index map so we never hard-code positions
    idx = {name: i for i, name in enumerate(names)}

    # Retrieve the three SALT2 photometric parameters by name
    x0 = params[idx["x0"]]
    x1 = params[idx["x1"]]
    c = params[idx["c"]]

    # Tripp distance modulus
    mB_star = -2.5 * np.log10(x0) + mb_const
    mu = mB_star - M + alpha * x1 - beta * c

    # Gradient of mu w.r.t. the parameters that enter it
    gradient = {
        "x0": -2.5 / (x0 * np.log(10)),  # d(m_B*)/d(x0)
        "x1": alpha,  # d(mu)/d(x1)
        "c": -beta,  # d(mu)/d(c)
    }
    mu_params = ["x0", "x1", "c"]

    # Add the z contribution only when z was a free parameter
    if "z" in idx:
        z = params[idx["z"]]
        if cosmo is None:
            from astropy.cosmology import FlatLambdaCDM

            cosmo = FlatLambdaCDM(H0=70, Om0=0.3)
        dz = 1e-5
        dL = cosmo.luminosity_distance(z).value
        dL_p = cosmo.luminosity_distance(z + dz).value
        dL_m = cosmo.luminosity_distance(z - dz).value
        gradient["z"] = (5.0 / np.log(10)) * (dL_p - dL_m) / (2.0 * dz * dL)
        mu_params.append("z")

    # Gradient vector g and covariance sub-matrix C_sub
    # restricted to the parameters that actually enter mu  (t0 excluded)
    g = np.array([gradient[p] for p in mu_params])
    sub_idx = np.array([idx[p] for p in mu_params])
    C_sub = cov[np.ix_(sub_idx, sub_idx)]

    # sigma_mu^2 = g^T . C_sub . g  (exact first-order error propagation)
    sigma_mu = float(np.sqrt(max(0.0, g @ C_sub @ g)))

    # Marginal 1-sigma errors on every fitted parameter
    sigmas = {name: float(np.sqrt(max(0.0, cov[i, i]))) for name, i in idx.items()}

    # Off-diagonal covariances for mu-entering pairs (upper triangle only)
    covariances = {}
    for ia, a in enumerate(mu_params):
        for ib, b in enumerate(mu_params):
            if ia < ib:
                covariances[f"{a}:{b}"] = float(C_sub[ia, ib])

    return {
        "mu": float(mu),
        "sigma_mu": sigma_mu,
        "gradient": gradient,
        "sigmas": sigmas,
        "covariances": covariances,
    }


print("computemustaterrors ready  (gradient g^T.C.g, param_names-aware)")

## 6 — Check SALT2 model and LSST band registration

In [ ]:
salt2_test = sncosmo.Model(source=SALT2_SOURCE)
print(f"Model       : {salt2_test.source.name}  v{salt2_test.source.version}")
print(f"Parameters  : {salt2_test.param_names}")
print(f"Phase range : {salt2_test.source.minphase():.1f} – {salt2_test.source.maxphase():.1f} days")
print(f"Wave range  : {salt2_test.source.minwave():.0f} – {salt2_test.source.maxwave():.0f} Å")
print(f"\nRedshift fit bounds: z ∈ [{Z_BOUND_LO}, {Z_BOUND_HI}]  (agnostic, no TNS z used)")
print("\nLSST band registration:")
for b in BAND_ORDER:
    bp = sncosmo.get_bandpass(BAND_PREFIX + b)
    print(f"  lsst{b}  λ_eff={bp.wave_eff:.0f} Å  ✓")

## 7 — Run two-pass free-z SALT2 fits

The TNS spec-z is displayed alongside for information only — it is not passed to the fitter.

In [ ]:
fit_results = {}  # diaObjectId → result dict
fit_errors = {}


for _, row in snia_catalog.iterrows():
    oid = int(row[obj_id_col])
    z_tns = float(row[tns_z_col])
    name = row.get(tns_name_col, str(oid))
    ttype = row.get(tns_type_col, "?")
    lc_df = lc_dict.get(oid, pd.DataFrame())

    if lc_df.empty:
        fit_results[oid] = {"success": False, "message": "No light curve data"}
        print(f"  {name:22s}  z_TNS={z_tns:.4f}  ✗  No data")
        continue

    # TNS z is NOT passed to the fitter
    res = fit_salt2_free_z_fink(lc_df)
    fit_results[oid] = res

    if res["success"]:
        dz = res["z"] - z_tns

        param_names = res["result"].vparam_names
        params = res["result"].parameters
        cov = res["result"].covariance
        fit_errors[oid] = computemustaterrors(param_names, params, cov)
        sigma_mu = fit_errors[oid]["sigma_mu"]

        print(
            f"  {name:22s}  [{ttype:10s}]  "
            f"z_TNS={z_tns:.4f}  z_fit={res['z']:.4f}±{res['z_err']:.4f}  "
            f"Δz={dz:+.4f}  "
            f"x1={res['x1']:+.3f}  c={res['c']:+.3f}  "
            f"χ²/dof={res['chi2_red']:.2f}  μ={res['mu_tripp']:.2f} σ_mu={sigma_mu:.3f}"
        )
    else:
        print(f"  {name:22s}  [{ttype:10s}]  ✗  {res['message']}")

n_ok = sum(1 for r in fit_results.values() if r["success"])
n_tot = len(fit_results)
print(f"\nSummary: {n_ok}/{n_tot} fits converged.")

## 8 — Per-event plot: light curve + χ²(z) grid scan

In [ ]:
def plot_event_lc_and_zgrid(oid: int, name: str, res: dict, z_tns: float):
    """Two-panel figure: multi-band LC with SALT2 model (left) + χ²(z) scan (right)."""
    fig, (ax_lc, ax_z) = plt.subplots(
        1,
        2,
        figsize=(13, 4),
        gridspec_kw={"width_ratios": [2, 1]},
    )

    # ── Left: light curve ─────────────────────────────────────────────────────
    if res.get("success"):
        obs = res["table"]
        fitted_model = res["fitted_model"]
        t0 = res["t0"]
        t_dense = np.linspace(float(obs["time"].min()) - 10, float(obs["time"].max()) + 10, 400)
        for b in BAND_ORDER:
            bname = BAND_PREFIX + b
            color = BAND_COLORS[b]
            mask = np.array(obs["band"]) == bname
            if mask.sum() > 0:
                ax_lc.errorbar(
                    obs["time"][mask] - t0,
                    obs["flux"][mask],
                    yerr=obs["fluxerr"][mask],
                    color=color,
                    fmt="o",
                    ms=4,
                    capsize=2,
                    label=b,
                    zorder=3,
                )
            try:
                f_model = fitted_model.bandflux(bname, t_dense, zp=RUBIN_ZP, zpsys=ZPSYS)
                valid = np.isfinite(f_model)
                if valid.sum() > 1:
                    ax_lc.plot(t_dense[valid] - t0, f_model[valid], color=color, lw=1.5, zorder=2)
            except Exception:
                pass
        ax_lc.axhline(0.0, color="k", lw=0.5, ls="--")
        pe = res["param_errors"]
        ax_lc.set_title(
            f"{name}  z_TNS={z_tns:.4f} (not used in fit)\n"
            f"z_fit={res['z']:.4f}±{pe.get('z', float('nan')):.4f}  "
            f"x1={res['x1']:+.2f}  c={res['c']:+.2f}  "
            f"χ²/dof={res['chi2_red']:.2f}",
            fontsize=8,
        )
        ax_lc.legend(fontsize=6, ncol=6, loc="upper right")
    else:
        ax_lc.set_title(f"{name}\nFit failed: {res.get('message', '')}", color="red", fontsize=8)
    ax_lc.set_xlabel(r"$t - t_0$ [days]", fontsize=9)
    ax_lc.set_ylabel("psfFlux [nJy]", fontsize=9)

    # ── Right: χ²(z) grid scan ────────────────────────────────────────────────
    z_grid = res.get("z_grid")
    chi2_grid = res.get("chi2_grid")
    if z_grid is not None and chi2_grid is not None:
        finite = np.isfinite(chi2_grid)
        ax_z.plot(z_grid[finite], chi2_grid[finite], "C0o-", ms=4, lw=1.2, label="grid χ² (Pass 1)")

        z_gb = res.get("z_grid_best", float("nan"))
        ax_z.axvline(z_gb, color="C0", ls="--", lw=1.2, label=f"grid best: {z_gb:.4f}")

        if res.get("success"):
            ax_z.axvline(res["z"], color="C1", ls="-", lw=1.8, label=f"z_fit: {res['z']:.4f}")

        # TNS spectroscopic z for reference only
        ax_z.axvline(z_tns, color="k", ls=":", lw=1.5, label=f"z_TNS (ref): {z_tns:.4f}")

        # Shade the Pass-2 refinement window
        lo = max(Z_BOUND_LO, z_gb - Z_REFINE_HALF)
        hi = min(Z_BOUND_HI, z_gb + Z_REFINE_HALF)
        ax_z.axvspan(lo, hi, alpha=0.12, color="C0", label="refine window")

        if finite.any():
            chi2_min = chi2_grid[finite].min()
            ax_z.set_ylim(chi2_min * 0.9, min(chi2_grid[finite].max(), chi2_min * 3 + 20))

        ax_z.legend(fontsize=7, loc="upper right")

    ax_z.set_xlabel("Redshift z", fontsize=9)
    ax_z.set_ylabel(r"$\chi^2$ (Pass 1, fixed z)", fontsize=9)
    ax_z.set_title("Coarse grid scan", fontsize=9)

    fig.tight_layout()
    plt.show()
    return fig

In [ ]:
ok_oids = [oid for oid in fit_results if fit_results[oid]["success"]]

if not ok_oids:
    print("No successful fits to plot.")
else:
    N_PLOT = min(len(ok_oids), 20)  # cap at 20 individual plots
    for oid in ok_oids[:N_PLOT]:
        meta = snia_catalog.loc[snia_catalog[obj_id_col] == oid].iloc[0]
        name = meta.get(tns_name_col, str(oid))
        z_tns = float(meta[tns_z_col])
        fig = plot_event_lc_and_zgrid(oid, name, fit_results[oid], z_tns)
        tns_safe = str(name).replace(" ", "_")
        fig.savefig(FIGS_DIR / f"lc_zgrid_{oid}_{tns_safe}.png", dpi=130, bbox_inches="tight")
        plt.show()
        plt.close(fig)

## 9 — Build results summary DataFrame

In [ ]:
rows = []
for oid, res in fit_results.items():
    if not res["success"]:
        continue
    meta = snia_catalog.loc[snia_catalog[obj_id_col] == oid].iloc[0]
    z_tns = float(meta[tns_z_col])
    z_fit = res["z"]
    dz = z_fit - z_tns
    rows.append(
        {
            "diaObjectId": oid,
            "tns_name": meta.get(tns_name_col, ""),
            "tns_type": meta.get(tns_type_col, ""),
            "z_tns": z_tns,
            "z_fit": z_fit,
            "z_err": res["z_err"],
            "z_grid_best": res["z_grid_best"],
            "dz": dz,
            "dz_1pz": dz / (1.0 + z_tns),
            "t0": res["t0"],
            "x0": res["x0"],
            "x1": res["x1"],
            "c": res["c"],
            "mB": res["mB"],
            "mu_tripp": res["mu_tripp"],
            "chi2_red": res["chi2_red"],
            "ndof": res["ndof"],
        }
    )

results_df = pd.DataFrame(rows)
print(f"{len(results_df)} successful fits.")
results_df.to_parquet(DATA_DIR / "salt2_freez_fit_results.parquet", index=False)
results_df.to_csv(DATA_DIR / "salt2_freez_fit_results.csv", index=False)

if len(results_df) > 0:
    print(
        results_df[["tns_name", "tns_type", "z_tns", "z_fit", "z_err", "dz", "dz_1pz", "chi2_red"]].to_string(
            index=False, float_format="{:.4f}".format
        )
    )

## 10 — ΛCDM distance modulus helpers

In [ ]:
C_LIGHT = 2.998e5  # km/s


def E_func(z, Om=OmegaM, Ol=OmegaL):
    return np.sqrt(Om * (1.0 + z) ** 3 + Ol)


def distance_modulus_lcdm(z_arr, h0=H0, Om=OmegaM, Ol=OmegaL):
    mu = np.zeros_like(z_arr, dtype=float)
    for i, z in enumerate(z_arr):
        if z <= 0:
            mu[i] = np.nan
            continue
        chi, _ = quad(lambda zp: 1.0 / E_func(zp, Om, Ol), 0.0, z)
        dl = (C_LIGHT / h0) * chi * (1.0 + z)
        mu[i] = 5.0 * np.log10(dl) + 25.0
    return mu


# Sanity check
print(f"ΛCDM μ(z=0.5) = {distance_modulus_lcdm(np.array([0.5]))[0]:.3f}  (expect ≈42.3)")

## 11 — Hubble diagram: μ(z_fit) and μ(z_TNS) vs ΛCDM

Two sets of points are shown:
- **Stars (red)**: μ computed from the fitted redshift `z_fit` (photometric z from LC).
- **Diamonds (orange)**: μ computed with the TNS spectroscopic redshift `z_TNS` as reference.

A connecting segment between the two symbols for the same object illustrates
how much the distance modulus shifts due to the redshift error.

In [ ]:
results_df

In [ ]:
if len(results_df) == 0:
    print("No successful fits — Hubble diagram cannot be drawn.")
else:
    # Quality cut
    hd = results_df[
        (results_df["chi2_red"] < 50.0) & (results_df["x1"].abs() < 4.0) & (results_df["c"].abs() < 0.4)
    ].copy()
    print(f"Hubble diagram: {len(hd)}/{len(results_df)} events after quality cuts.")

    # Compute μ from z_tns (using same SALT2 fit params, just with z_tns)
    SALT2_ZP_INT = 10.635
    hd["mB_from_x0"] = -2.5 * np.log10(hd["x0"].clip(lower=1e-30)) + SALT2_ZP_INT
    hd["mu_tns"] = hd["mB_from_x0"] - M_B + ALPHA * hd["x1"] - BETA * hd["c"]

    # ΛCDM theory curves
    z_theory = np.linspace(1e-3, Z_MAX_SEL * 1.05, 300)
    mu_lcdm = distance_modulus_lcdm(z_theory)

    # ── Figure ────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(
        2,
        1,
        figsize=(9, 7.5),
        gridspec_kw={"height_ratios": [3, 1]},
        sharex=True,
    )
    ax_hub = axes[0]
    ax_res = axes[1]

    # ΛCDM
    ax_hub.plot(
        z_theory,
        mu_lcdm,
        color="royalblue",
        lw=2.0,
        label=rf"ΛCDM ($\Omega_m={OmegaM}$, $\Omega_\Lambda={OmegaL}$, H₀={H0})",
    )

    if len(hd) > 0:
        # Connecting segments z_tns → z_fit
        for _, row in hd.iterrows():
            ax_hub.plot(
                [row["z_tns"], row["z_fit"]],
                [row["mu_tns"], row["mu_tripp"]],
                color="gray",
                lw=0.7,
                alpha=0.5,
                zorder=1,
            )

        # μ at z_tns (reference)
        ax_hub.scatter(
            hd["z_tns"],
            hd["mu_tns"],
            s=60,
            marker="D",
            color="orange",
            edgecolors="darkorange",
            lw=0.7,
            zorder=4,
            label=rf"μ(z_TNS spec)  N={len(hd)}",
        )
        # μ at z_fit
        ax_hub.scatter(
            hd["z_fit"],
            hd["mu_tripp"],
            s=60,
            marker="*",
            color="tomato",
            edgecolors="darkred",
            lw=0.5,
            zorder=5,
            label=rf"μ(z_fit photo)  N={len(hd)}",
        )

    ax_hub.set_ylabel(r"Distance modulus $\mu$", fontsize=12)
    ax_hub.legend(fontsize=9)
    ax_hub.set_title(
        "Hubble diagram — Fink/LSST TNS SNIa\nSALT2 fit with free z (stars) vs z_TNS reference (diamonds)",
        fontsize=11,
    )
    ax_hub.grid(True, alpha=0.3)

    # Residuals panel: Δμ = μ_fit − μ_ΛCDM(z_fit)
    ax_res.axhline(0.0, color="royalblue", lw=1.5)
    if len(hd) > 0:
        mu_lcdm_at_zfit = distance_modulus_lcdm(hd["z_fit"].values)
        ax_res.scatter(
            hd["z_fit"],
            hd["mu_tripp"].values - mu_lcdm_at_zfit,
            s=50,
            marker="*",
            color="tomato",
            edgecolors="darkred",
            lw=0.5,
            zorder=5,
        )
        mu_lcdm_at_ztns = distance_modulus_lcdm(hd["z_tns"].values)
        ax_res.scatter(
            hd["z_tns"],
            hd["mu_tns"].values - mu_lcdm_at_ztns,
            s=50,
            marker="D",
            color="orange",
            edgecolors="darkorange",
            lw=0.5,
            zorder=4,
        )
        ax_res.axhline(0.0, color="k", lw=0.8, ls="--")

    ax_res.set_xlabel("Redshift z", fontsize=12)
    ax_res.set_ylabel(r"$\Delta\mu$ [mag]", fontsize=12)
    ax_res.set_ylim(-2.5, 2.5)
    ax_res.grid(True, alpha=0.3)

    fig.tight_layout()
    fig.savefig(FIGS_DIR / "hubble_diagram_freez.png", dpi=150, bbox_inches="tight")
    print(f"Hubble diagram saved → {FIGS_DIR}/hubble_diagram_freez.png")
    plt.show()

## 12 — Photometric z accuracy: z_fit vs z_TNS

Three panels (same layout as `02b_elasticc2_fitsalt2andredshift_lightcurves.ipynb`):
1. **Main**: `z_fit ± σ_z` vs `z_TNS`, colour-coded by χ²/dof.
2. **Δz residuals**: `z_fit − z_TNS` vs `z_TNS`.
3. **Normalised**: `Δz / (1 + z_TNS)` vs `z_TNS` — the standard photo-z figure of merit.

In [ ]:
if len(results_df) < 2:
    print("Not enough successful fits for z_fit vs z_TNS diagnostic.")
else:
    df_ok = results_df.copy()

    z_true_arr = df_ok["z_tns"].values
    z_fit_arr = df_ok["z_fit"].values
    z_err_arr = df_ok["z_err"].values
    chi2_arr = df_ok["chi2_red"].values
    dz_arr = df_ok["dz"].values
    dz_1pz_arr = df_ok["dz_1pz"].values

    Z_ERR_MAX = 0.3
    z_err_display = np.clip(z_err_arr, 0, Z_ERR_MAX)

    rms_dz = float(np.std(dz_arr))
    rms_dz1pz = float(np.std(dz_1pz_arr))
    bias_dz = float(np.median(dz_arr))

    cmap = plt.cm.viridis_r
    norm = matplotlib.colors.Normalize(vmin=0, vmax=5)
    colors = cmap(norm(np.clip(chi2_arr, 0, 5)))

    fig9, axes9 = plt.subplots(
        3,
        1,
        figsize=(8, 13),
        sharex=True,
        gridspec_kw={"height_ratios": [3, 1.4, 1.4]},
    )

    # ── Panel 1: z_fit ± σ_z  vs  z_TNS ──────────────────────────────────────
    ax1 = axes9[0]
    for i in range(len(z_true_arr)):
        ax1.errorbar(
            z_true_arr[i],
            z_fit_arr[i],
            yerr=z_err_display[i],
            fmt="o",
            ms=6,
            color=colors[i],
            ecolor=colors[i],
            alpha=0.8,
            capsize=3,
            lw=0.8,
            zorder=3,
        )

    z_ref = np.linspace(
        max(Z_BOUND_LO, z_true_arr.min() - 0.02), min(Z_BOUND_HI, z_true_arr.max() + 0.02), 200
    )
    ax1.plot(z_ref, z_ref, "k--", lw=1.5, label=r"$z_{\rm fit} = z_{\rm TNS}$")
    ax1.fill_between(z_ref, z_ref - rms_dz, z_ref + rms_dz, alpha=0.10, color="k", label=f"±RMS={rms_dz:.4f}")

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cb = fig9.colorbar(sm, ax=ax1, pad=0.02, fraction=0.04)
    cb.set_label(r"$\chi^2$/dof", fontsize=10)

    ax1.set_ylabel(r"$z_{\rm fit}$ (SALT2, free $z$)", fontsize=12)
    ax1.set_title(
        f"Photometric redshift — SALT2 free-z fit on Fink/LSST TNS SNIa\n"
        f"N={len(df_ok)} events  |  "
        f"RMS Δz={rms_dz:.4f}  |  "
        f"RMS Δz/(1+z)={rms_dz1pz:.4f}  |  "
        f"bias={bias_dz:+.4f}\n"
        f"Model: {SALT2_SOURCE}  |  σ_z capped at {Z_ERR_MAX} for display",
        fontsize=10,
    )
    ax1.legend(fontsize=9, loc="upper left")
    ax1.grid(True, alpha=0.3)

    # ── Panel 2: Δz = z_fit − z_TNS ──────────────────────────────────────────
    ax2 = axes9[1]
    for i in range(len(z_true_arr)):
        ax2.errorbar(
            z_true_arr[i],
            dz_arr[i],
            yerr=z_err_display[i],
            fmt="o",
            ms=5,
            color=colors[i],
            ecolor=colors[i],
            alpha=0.8,
            capsize=3,
            lw=0.8,
            zorder=3,
        )
    ax2.axhline(0.0, color="k", lw=1.2, ls="--")
    ax2.axhline(+rms_dz, color="gray", lw=0.8, ls=":")
    ax2.axhline(-rms_dz, color="gray", lw=0.8, ls=":")

    # Binned median
    try:
        z_bins = np.linspace(z_true_arr.min(), z_true_arr.max(), 6)
        z_bin_cen = 0.5 * (z_bins[:-1] + z_bins[1:])
        dz_med = [
            np.median(dz_arr[(z_true_arr >= z_bins[k]) & (z_true_arr < z_bins[k + 1])])
            for k in range(len(z_bin_cen))
        ]
        ax2.plot(z_bin_cen, dz_med, "rs-", ms=6, lw=1.5, label="binned median", zorder=5)
        ax2.legend(fontsize=8)
    except Exception:
        pass

    ax2.set_ylabel(r"$\Delta z = z_{\rm fit} - z_{\rm TNS}$", fontsize=12)
    ax2.set_title(f"Redshift residuals  |  RMS={rms_dz:.4f}  bias={bias_dz:+.4f}", fontsize=9)
    ax2.grid(True, alpha=0.3)

    # ── Panel 3: Δz / (1 + z_TNS) ─────────────────────────────────────────────
    ax3 = axes9[2]
    for i in range(len(z_true_arr)):
        ax3.errorbar(
            z_true_arr[i],
            dz_1pz_arr[i],
            yerr=z_err_display[i] / (1.0 + z_true_arr[i]),
            fmt="o",
            ms=5,
            color=colors[i],
            ecolor=colors[i],
            alpha=0.8,
            capsize=3,
            lw=0.8,
            zorder=3,
        )
    ax3.axhline(0.0, color="k", lw=1.2, ls="--")
    ax3.axhline(+rms_dz1pz, color="gray", lw=0.8, ls=":", label=f"±RMS={rms_dz1pz:.4f}")
    ax3.axhline(-rms_dz1pz, color="gray", lw=0.8, ls=":")

    try:
        dz1pz_med = [
            np.median(dz_1pz_arr[(z_true_arr >= z_bins[k]) & (z_true_arr < z_bins[k + 1])])
            for k in range(len(z_bin_cen))
        ]
        ax3.plot(z_bin_cen, dz1pz_med, "rs-", ms=6, lw=1.5, label="binned median", zorder=5)
        ax3.legend(fontsize=8)
    except Exception:
        pass

    ax3.set_xlabel(r"$z_{\rm TNS}$ (spectroscopic reference)", fontsize=12)
    ax3.set_ylabel(r"$\Delta z\,/\,(1+z_{\rm TNS})$", fontsize=12)
    ax3.set_title(
        rf"Normalised residuals  |  RMS={rms_dz1pz:.4f}  "
        r"$\equiv$ photo-$z$ figure of merit",
        fontsize=9,
    )
    ax3.grid(True, alpha=0.3)

    fig9.tight_layout()
    fig9.savefig(FIGS_DIR / "zfit_vs_ztns.png", dpi=150, bbox_inches="tight")
    print(f"Photo-z diagnostic saved → {FIGS_DIR}/zfit_vs_ztns.png")
    plt.show()

    print(f"\nz_fit performance summary:")
    print(f"  N events              : {len(df_ok)}")
    print(f"  RMS Δz                : {rms_dz:.4f}")
    print(f"  RMS Δz/(1+z)          : {rms_dz1pz:.4f}")
    print(f"  Median bias Δz        : {bias_dz:+.4f}")
    print(f"  Median |Δz|           : {np.median(np.abs(dz_arr)):.4f}")
    print(f"  Fraction |Δz/(1+z)|<0.06 : {(np.abs(dz_1pz_arr) < 0.06).mean():.1%}")

## 13 — Distribution of redshift residuals and σ_z

In [ ]:
if len(results_df) >= 2:
    df_ok = results_df.copy()
    Z_ERR_MAX = 0.3

    fig10, axes10 = plt.subplots(1, 3, figsize=(14, 4))

    # σ_z histogram
    vals_zerr = df_ok["z_err"].clip(upper=Z_ERR_MAX)
    axes10[0].hist(vals_zerr, bins=15, color="C0", edgecolor="white")
    axes10[0].axvline(
        np.median(vals_zerr), color="k", ls="--", lw=1.5, label=f"median={np.median(vals_zerr):.4f}"
    )
    axes10[0].set_xlabel(r"$\sigma_z$ from fit (capped at 0.3)", fontsize=10)
    axes10[0].set_ylabel("N events", fontsize=10)
    axes10[0].set_title(f"σ_z distribution  RMS={vals_zerr.std():.4f}", fontsize=9)
    axes10[0].legend(fontsize=8)

    # Δz histogram
    axes10[1].hist(df_ok["dz"], bins=15, color="C1", edgecolor="white")
    axes10[1].axvline(
        np.median(df_ok["dz"]), color="k", ls="--", lw=1.5, label=f"median={np.median(df_ok['dz']):+.4f}"
    )
    axes10[1].axvline(0.0, color="r", ls=":", lw=1.0)
    axes10[1].set_xlabel(r"$\Delta z = z_{\rm fit} - z_{\rm TNS}$", fontsize=10)
    axes10[1].set_ylabel("N events", fontsize=10)
    axes10[1].set_title(f"Δz  RMS={df_ok['dz'].std():.4f}", fontsize=9)
    axes10[1].legend(fontsize=8)

    # Δz/(1+z) histogram
    axes10[2].hist(df_ok["dz_1pz"], bins=15, color="C2", edgecolor="white")
    axes10[2].axvline(
        np.median(df_ok["dz_1pz"]),
        color="k",
        ls="--",
        lw=1.5,
        label=f"median={np.median(df_ok['dz_1pz']):+.5f}",
    )
    axes10[2].axvline(0.0, color="r", ls=":", lw=1.0)
    axes10[2].set_xlabel(r"$\Delta z\,/\,(1+z_{\rm TNS})$", fontsize=10)
    axes10[2].set_ylabel("N events", fontsize=10)
    axes10[2].set_title(f"Δz/(1+z)  RMS={df_ok['dz_1pz'].std():.5f}", fontsize=9)
    axes10[2].legend(fontsize=8)

    fig10.suptitle(f"Redshift recovery — SALT2 free-z on Fink/LSST TNS SNIa  (N={len(df_ok)})", fontsize=11)
    fig10.tight_layout()
    fig10.savefig(FIGS_DIR / "zfit_residuals_histograms.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Not enough successful fits for residual histograms.")

## 14 — SALT2 parameter distributions (x1, c, χ²/dof)

In [ ]:
if len(results_df) >= 2:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].hist(results_df["x1"], bins=15, color="steelblue", edgecolor="white")
    axes[0].axvline(0.0, color="k", ls="--", lw=1)
    axes[0].set_xlabel(r"$x_1$ (stretch)", fontsize=12)
    axes[0].set_ylabel("Count", fontsize=12)
    axes[0].set_title(r"SALT2 stretch $x_1$")

    axes[1].hist(results_df["c"], bins=15, color="tomato", edgecolor="white")
    axes[1].axvline(0.0, color="k", ls="--", lw=1)
    axes[1].set_xlabel(r"$c$ (colour)", fontsize=12)
    axes[1].set_ylabel("Count", fontsize=12)
    axes[1].set_title(r"SALT2 colour $c$")

    axes[2].hist(results_df["chi2_red"], bins=15, color="seagreen", edgecolor="white")
    axes[2].axvline(1.0, color="k", ls="--", lw=1, label="χ²/dof = 1")
    axes[2].set_xlabel(r"$\chi^2/\mathrm{dof}$", fontsize=12)
    axes[2].set_ylabel("Count", fontsize=12)
    axes[2].set_title(r"Reduced $\chi^2$")
    axes[2].legend()

    fig.suptitle("SALT2 free-z fit parameters — Fink/LSST TNS SNIa", fontsize=13)
    fig.tight_layout()
    fig.savefig(FIGS_DIR / "salt2_param_distributions.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Not enough fits for parameter distributions.")

## 15 — Summary table

In [ ]:
if len(results_df) > 0:
    display_cols = [
        "tns_name",
        "tns_type",
        "z_tns",
        "z_fit",
        "z_err",
        "dz",
        "dz_1pz",
        "x1",
        "c",
        "mB",
        "mu_tripp",
        "chi2_red",
    ]
    print(results_df[display_cols].to_string(index=False, float_format="{:.4f}".format))
else:
    print("No successful fits.")

---
## Notes and caveats

### Parameter status

| Parameter | NB 03 (z fixed) | This notebook (z free) |
|-----------|----------------|-------------------------|
| `z`  | **fixed** to `f:xm_tns_redshift` | **free** — fitted from LC shape |
| `t0` | free | free |
| `x0` | free | free |
| `x1` | free | free |
| `c`  | free | free |

### Two-pass fitting strategy

| Pass | Free params | Purpose |
|------|-------------|--------|
| 1 — coarse grid (`N_GRID_Z` pts) | `t0, x0, x1, c` (z fixed on grid) | Locate global χ² minimum in z |
| 2 — refined fit | `z, t0, x0, x1, c` | Precise parameters + covariance → σ_z |

### Key limitations

- **z–c degeneracy**: both redshift and colour shift observed broadband SEDs.
  Light-curve-only photometric redshifts are inherently imprecise (σ_z/(1+z) ~ 0.05–0.15
  for typical Rubin Year 1 sampling).
- **σ_z underestimate**: the covariance from `sncosmo.fit_lc` is the statistical
  uncertainty only; systematic errors (template mismatch, dust, host contamination)
  are not captured.
- **Light-curve sampling**: early Rubin data may lack the pre-maximum coverage
  needed for a well-constrained fit.
- **ZP convention**: psfFlux in nJy, ZP = 31.4, zpsys = 'ab' (Rubin AB definition).
- **Tripp calibration**: α, β, M_B fixed to Betoule+ 2014 values.
